In [1]:
from pipeline.src.clean.cleaners import EventsCleaner
from pipeline.src.clean.utils import extract_location_parts, parse_percentage
import pandas as pd
import numpy as np

In [2]:
input_series = pd.Series([
    "14%",
    "0%",
    "100%",
    "-5%",
    " 42% ",
    "---",
    "",
    "abc%",
    None
])

In [4]:
result = parse_percentage(input_series)


In [8]:
actual = result

In [7]:
actual

np.float64(14.0)

In [2]:
def sample_events_df():
    data = {
        "date": ["2021-01-01", "invalid", "2022-12-31"],
        "location": [
            "Sydney, New South Wales, Australia",  # Three parts
            "Macau, China",                         # Two parts
            "Unknown"                               # One part
        ]
    }
    return pd.DataFrame(data)

In [4]:
loc_series = pd.Series(["Macau, China"])
parts = extract_location_parts(loc_series)

In [11]:
cleaner = EventsCleaner(sample_events_df())
cleaned_df = cleaner.clean()

In [19]:
extract_location_parts(sample_events_df()['location'])

,city,state,country
0,Sydney,New South Wales,Australia
1,Macau,China,None
2,Unknown,None,None


In [22]:
sample_events_df()['location']

0    Sydney, New South Wales, Australia
1                          Macau, China
2                               Unknown
Name: location, dtype: object

In [20]:
extract_location_parts(pd.Series(["Macau, China"]))

,city,state,country
0,Macau,NaN,China


In [25]:
extract_location_parts(pd.Series(["Macau, China","Macau, China","Sydney, New South Wales, Australia",]))

,city,state,country
0,Macau,China,None
1,Macau,China,None
2,Sydney,New South Wales,Australia


In [26]:
def extract_location_parts2(location_series: pd.Series) -> pd.DataFrame:
    """
    Given a Series of location strings (e.g. "City, State, Country" or "City, Country"),
    split them into components in a vectorized manner.
    
    For each location string:
      - If there are three or more non-empty parts, assign:
          city = first part, state = second part, country = last part.
      - If there are exactly two parts, assign:
          city = first part, state = NaN, country = second part.
      - Otherwise, return NaN for missing values.
    """
    def split_location(loc):
        # Split by comma, strip whitespace, and filter out empty parts.
        parts = [p.strip() for p in loc.split(',') if p.strip() != ""]
        if len(parts) >= 3:
            return pd.Series({"city": parts[0], "state": parts[1], "country": parts[-1]})
        elif len(parts) == 2:
            return pd.Series({"city": parts[0], "state": np.nan, "country": parts[1]})
        else:
            return pd.Series({"city": parts[0] if parts else np.nan, "state": np.nan, "country": np.nan})
    
    return location_series.apply(split_location)

In [30]:
extract_location_parts2(pd.Series(["Macau, China","Macau, China","Sydney, New South Wales, Australia",]))

,city,state,country
0,Macau,NaN,China
1,Macau,NaN,China
2,Sydney,New South Wales,Australia
